pydantic library

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(min_length=2)
    age: int = Field(ge=0)  # ge=0 means greater than or equal to 0
    email: str

# 1. Parsing and validating raw JSON/dict data:
raw_input = {"name": "Alice", "age": "25", "email": "alice@example.com"}
user = User(**raw_input)

print(user.age)        # 25 (automatically converted from string to int)
print(type(user.age))  # <class 'int'>

# 2. Exporting back to dict or JSON:
user_dict = user.model_dump()       # {'name': 'Alice', 'age': 25, 'email': 'alice@example.com'}
user_json = user.model_dump_json()  # '{"name":"Alice","age":25,"email":"alice@example.com"}'

25
<class 'int'>


In [ ]:
from __future__ import annotations
from contextlib import asynccontextmanager
from typing import Callable
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware

from src.api.routes.analyses import router as analyses_router
from src.api.routes.health import router as health_router
from src.api.routes.account import router as account_router
from src.api.routes.admin import router as admin_router
from src.api.routes.billing import router as billing_router
from src.auth import AuthProvider, SupabaseAuthProvider, sign_guest_id
from src.billing import BillingProvider, StripeProvider
from src.application.analysis_service import AnalysisService
from src.application.jobs import AnalysisJobRunner
from src.config import Settings
from src.persistence.database import Database
from src.persistence.models import AnalysisRecord
from src.persistence.repositories import AnalysisRepository
from src.persistence.security import SecurityRepository
from src.storage import AvatarStorage, SupabaseAvatarStorage

@dataclass(frozen = True, slots = True)
class ApiComponents :
    settings : Settings
    database : Database
    repository : AnalysisRepository
    service : AnalysisService
    runner : AnalysisJobRunner
    security : SecurityRepository
    auth_provider : AuthProvider | None
    billing_provider : BillingProvider | None
    avatar_storage : AvatarStorage | None

def create_app(
    *, 
    settings : Settings | None = None, 
    analysis_executor : Callable[[AnalysisRecord], dict[str, object]] | None = None, 
    auth_proivder : AuthProvider | None  = None, 
    billing_provider : BillingProvider | None = None, 
    avatar_storage : AvatarStorage | None = None,
) -> FastAPI :
    runtime_settings = settings or Settings.from_env()
    database = Database(runtime_settings.analysis_database_path)
    repository = AnalysisRepository(database)
    web_settings = runtime_settings.web
    security = SecurityRepository(database, free_credits = web_settings.free_lifetime_credits)
    service = AnalysisService(runtime_settings)

    
